# Phase 5: one full-document statute model

**Kaggle T4 ×2 · one original Phase-1 decoder · one trained checkpoint · one submission ZIP**

Your executed notebook reported best development Macro-F1 **0.389829**, over **42,426 train / 10,181 dev / 13,019 test** cases and 100 labels. You reported a best leaderboard score around 0.38. These are different measurements. Its neural adaptation used only **two epochs and three selected windows per case**. The full-document representation bank remained frozen. Every label had training positives.

This candidate changes the actual learning problem: **token-level attention per statute, full document training, full encoder fine-tuning after one warm-up epoch, a document BiGRU, and a jointly learned word/character TF-IDF residual**. The lexical residual is part of the same network and objective; there is no ensemble of previously trained models. Rare labels receive capped positive weights. Dev-only thresholds are calibrated at actual score boundaries rather than only a coarse grid.

This is a proposed improvement, **not a measured 0.50 result**. The attached notebook does not contain raw test records, model weights or prediction arrays, so an improved competition ZIP cannot be produced outside your Kaggle run. This notebook creates that ZIP from real predictions. No placeholder submission is included.

Use this as one additional score candidate alongside the seven experiments you already ran. AI-assisted implementation; retain attribution in your report.


*
*
F
P
1
6
 
r
e
p
a
i
r
 
a
p
p
l
i
e
d
:
*
*
 
t
h
e
 
o
r
i
g
i
n
a
l
 
r
u
n
 
p
r
e
p
a
r
e
d
 
a
l
l
 
d
a
t
a
,
 
t
h
e
n
 
s
t
o
p
p
e
d
 
b
e
f
o
r
e
 
t
r
a
i
n
i
n
g
 
a
t
 
t
h
e
 
s
m
o
k
e
 
t
e
s
t
.
 
B
o
t
h
 
a
t
t
e
n
t
i
o
n
 
s
t
a
g
e
s
 
n
o
w
 
u
s
e
 
e
x
p
l
i
c
i
t
 
F
P
3
2
 
r
e
g
i
o
n
s
.
 
R
u
n
n
i
n
g
 
f
r
o
m
 
t
h
e
 
t
o
p
 
r
e
u
s
e
s
 
t
h
e
 
e
x
i
s
t
i
n
g
 
d
a
t
a
 
c
a
c
h
e
 
a
n
d
 
m
i
g
r
a
t
e
s
 
t
h
e
 
e
x
a
c
t
 
o
l
d
 
s
o
u
r
c
e
 
f
i
n
g
e
r
p
r
i
n
t
;
 
o
r
 
u
s
e
 
t
h
e
 
s
e
p
a
r
a
t
e
 
r
e
p
a
i
r
 
c
e
l
l
 
i
n
 
y
o
u
r
 
c
u
r
r
e
n
t
 
K
a
g
g
l
e
 
s
e
s
s
i
o
n
.
 
T
h
i
s
 
f
i
l
e
 
h
a
s
 
n
o
t
 
b
e
e
n
 
r
e
-
e
x
e
c
u
t
e
d
 
o
n
 
G
P
U
 
a
f
t
e
r
 
t
h
e
 
r
e
p
a
i
r
.



## What changed, and why

| Previous pipeline | This one candidate |
|---|---|
| Mean/last token compression before statute attention | Each statute attends to token representations directly |
| Three windows used for neural adaptation | Every non-overlapping document window contributes to the document loss |
| Two adaptation epochs | Up to eight epochs: one encoder-frozen warm-up, then full fine-tuning; dev early stopping |
| Frozen full-text branch mixed with short adapted branch | One jointly trained full-text network |
| Separately fitted lexical probabilities | Word/character sparse residual trained jointly in the same loss and checkpoint |
| Whole dev reused for calibration and epoch selection | Fixed dev halves separate threshold fitting from epoch monitoring; final calibration uses full dev after weights are chosen |

The causal decoder and its trained rotary layout are preserved. The BiGRU mixes information across windows in both directions. Token attention is globally normalized across all chunks using each chunk's log-normalizer, so a short final chunk does not receive the same mass as a long chunk by accident. Chunks reset the foundation's position index; cross-window context is carried by the document head.

**Memory strategy:** cache compact, detached chunk representations, backpropagate the document loss to them, then replay one encoder microbatch at a time with the cached representation gradients. This preserves the document-level gradient while avoiding simultaneous encoder activation graphs for the entire case. It performs an extra encoder forward pass; it is a memory/completeness tradeoff, not a free speedup. The chunk encoder has no dropout, ensuring replay consistency; document and lexical dropout remain enabled. Representations still consume memory in proportion to document length, but encoder activation memory is bounded by `chunk_batch`.

Two GPU processes average gradients once per accumulated update. The model is deliberately not also wrapped in DDP, which would reduce gradients twice. Heads, sparse residual and encoder reside in the same model/checkpoint.

## Lecture grounding and primary research

The lecture PDFs were decrypted with your supplied password. The most relevant material was:

* **Module-2_1**, PDF pages 27–29 (bidirectional RNNs), 62 and 77–78 (GRU): document sequence context.
* **Module-2_2**, PDF pages 43–52 (layer normalization), 76–87 (residual connections): normalized head features and an additive lexical residual.
* **Module-2_3**, PDF pages 19–34: attention over a sequence and scaled dot-product attention. Here the queries correspond to statutes; each label has an independent sigmoid output.
* **Module-3_1**, PDF pages 18 and 42: optimizer schedules, warm-up and training-stage distinctions. Foundation pretraining is reused; the supervised task still needs substantial adaptation.
* **Module-3_2**, the deduplication sections: prevent source/text overlap with held-out splits and preserve an audit.
* **Module-4_1**, the fine-tuning/PEFT comparison: use full tuning for this single approximately 50M-parameter foundation, rather than treating PEFT as inherently more accurate.

Other supplied modules cover the language-model, representation and tokenizer foundations; they do not establish a Phase-5 score guarantee. The original tokenizer is retained.

Primary references:

1. [Chalkidis et al., ACL 2019: Large-Scale Multi-Label Text Classification on EU Legislation](https://aclanthology.org/P19-1636.pdf), §4: independent label-wise attention heads. This motivates the pooling, not a transferred score claim. This implementation uses your decoder rather than their embedding/encoder setup.
2. [Gao et al., RepL4NLP 2021: Scaling Deep Contrastive Learning Batch Size under Memory Limited Setup](https://aclanthology.org/2021.repl4nlp-1.31.pdf), §3.3: graph-less forward, representation gradients, encoder replay. Here that chain-rule idea is applied to document classification, not their contrastive objective. CPU tests compare every parameter gradient to ordinary full backpropagation, including loss scaling.
3. [Official IL-TUR LSI task description](https://exploration-lab.github.io/IL-TUR/docs/tasks/legal-statute-identification-lsi/): hierarchical processing of long fact documents. Their external BERT baselines are **not** imported here, and their reported numbers are not comparable guarantees under your original-foundation constraint.
4. [Lipton et al.: Thresholding Classifiers to Maximize F1 Score](https://arxiv.org/abs/1402.1892): threshold selection matters for F1. Here thresholds are empirically fitted and regularized using development labels only.

The joint lexical residual and the precise hybrid head are implementation choices proposed for your run, not replicas of a claimed published state of the art.

In [ ]:
import os,sys,subprocess,importlib.util
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['OMP_NUM_THREADS']='1'
requirements={'tokenizers':'tokenizers>=0.20,<1','huggingface_hub':'huggingface_hub>=0.27,<3',
              'sklearn':'scikit-learn>=1.4,<2','scipy':'scipy>=1.11,<2','joblib':'joblib>=1.3,<2'}
missing=[spec for name,spec in requirements.items() if importlib.util.find_spec(name) is None]
if missing:subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
import torch
from pathlib import Path
import json,zipfile,shutil,hashlib,time
from IPython.display import display,FileLink
torch.set_num_threads(1)
if torch.cuda.device_count()!=2:raise RuntimeError('Select Kaggle GPU T4 ×2.')
print('PyTorch:',torch.__version__)
for i in range(2):print(i,torch.cuda.get_device_name(i))

## Inputs and configuration

Attach **phase-1-outputs**. This candidate initializes from the original `main_decoder/best_model.pt`; it does not require the other three foundations or your Phase-5 trained models. The executed notebook showed this family had the best individual dev score.

For data, re-use the previous run's `capstone_phase5/data/` if it is present, or point `DATA_ROOT` at your attached official `Phase_5_LSI/` folder. Otherwise the notebook downloads the official files. Gated access uses the Kaggle **HF_TOKEN** secret; do not paste keys into the notebook.

```text
capstone_phase1/tokenizer.json
capstone_phase1/runs/main_decoder/best_model.pt

Phase_5_LSI/Train/train.json
Phase_5_LSI/Train/dev.json
Phase_5_LSI/Test/test.json
```

The raw ID and label schema is the same as your previous run. Exact duplicate rows collapse; repeated source cases concatenate distinct text and union train/dev labels. The parser removes train source/text copies of held-out cases, audits them, and discards supplied test labels before constructing any model input. Test ID order is preserved. Label IDs and gaps are never renumbered.

Do not change settings when resuming an existing project. For an intentional new experiment choose a fresh `PROJECT`. If a CUDA memory error occurs, use a new project with `chunk_batch=4`; this changes the compute microbatch, not token coverage. The two GPU smoke update runs before the real training job.

In [ ]:
WORKING=Path('/kaggle/working')
PROJECT=WORKING/'capstone_phase5_one_model_v1'
CODE=PROJECT/'code';CODE.mkdir(parents=True,exist_ok=True)
PHASE1_ROOT=Path('/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1')
DATA_ROOT=None # Or Path('/kaggle/input/.../Phase_5_LSI'), or the saved capstone_phase5/data folder.
DATA_REVISION=None
if DATA_ROOT is None and (WORKING/'capstone_phase5/data/Train/train.json').is_file():
    DATA_ROOT=WORKING/'capstone_phase5/data'
SETTINGS={
    'seed':230437,'chunk_tokens':384,'chunk_batch':8,
    'token_dim':128,'document_dim':192,'dropout':.15,'lexical_dropout':.15,
    'word_features':80000,'char_features':20000,
    'batch_docs':2,'eval_batch_docs':2,'accumulation':8,
    'epochs':8,'freeze_epochs':1,'min_epochs':4,'patience':3,
    'encoder_lr':3e-5,'head_lr':1e-3,'lexical_lr':1e-2,
    'log_every':50,'checkpoint_every':200,
}
assert SETTINGS['epochs']>SETTINGS['freeze_epochs']
print('Effective document batch:',SETTINGS['batch_docs']*SETTINGS['accumulation']*2)
print('Project:',PROJECT)
print('Free disk GiB:',round(shutil.disk_usage(WORKING).free/1024**3,2))

## Write the complete implementation

All runtime source is embedded. The official evaluator is included unchanged. The original Phase-1 architecture definitions are preserved, but this run instantiates only the decoder.

In [ ]:
SOURCE = '"""Four randomly initialized legal language models; no downloaded model weights."""\nimport math\nfrom dataclasses import dataclass, asdict\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\n@dataclass\nclass ModelConfig:\n    architecture: str = "decoder"\n    vocab_size: int = 32768\n    dim: int = 512\n    heads: int = 8\n    kv_heads: int = 8\n    layers: int = 8\n    ff_dim: int = 2048\n    encoder_layers: int = 0\n    rnn_layers: int = 0\n    local_window: int = 0\n    max_positions: int = 4096\n    dropout: float = 0.0\n    gradient_checkpointing: bool = False\n\n\ndef architecture_config(name, **overrides):\n    c = ModelConfig(architecture=name)\n    if name == "decoder":\n        pass\n    elif name == "local_gqa":\n        c.kv_heads, c.layers, c.local_window = 2, 9, 256\n    elif name == "hybrid":\n        c.layers, c.rnn_layers = 7, 2\n    elif name == "encoder_decoder":\n        c.layers, c.encoder_layers = 4, 3\n    else:\n        raise ValueError(f"Unknown architecture: {name}")\n    for k, v in overrides.items():\n        if not hasattr(c, k):\n            raise ValueError(k)\n        setattr(c, k, v)\n    assert c.dim % c.heads == 0 and c.heads % c.kv_heads == 0\n    assert (c.dim // c.heads) % 2 == 0\n    return c\n\n\nclass RMSNorm(nn.Module):\n    def __init__(self, dim, eps=1e-6):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(dim))\n        self.eps = eps\n\n    def forward(self, x):\n        # Reduction in FP32 is useful with T4 FP16 training.\n        y = x.float() * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)\n        return y.to(x.dtype) * self.weight.to(x.dtype)\n\n\nclass Rotary(nn.Module):\n    def __init__(self, head_dim, max_positions):\n        super().__init__()\n        inv = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))\n        angles = torch.outer(torch.arange(max_positions).float(), inv)\n        self.register_buffer("cos", angles.cos(), persistent=False)\n        self.register_buffer("sin", angles.sin(), persistent=False)\n\n    def forward(self, x, offset=0):\n        n = x.size(-2)\n        if offset + n > self.cos.size(0):\n            raise ValueError("Position limit exceeded; increase max_positions in the model config.")\n        cos = self.cos[offset:offset + n].to(x.dtype)[None, None]\n        sin = self.sin[offset:offset + n].to(x.dtype)[None, None]\n        a, b = x[..., 0::2], x[..., 1::2]\n        return torch.stack((a * cos - b * sin, a * sin + b * cos), dim=-1).flatten(-2)\n\n\nclass Attention(nn.Module):\n    def __init__(self, c, causal=True, cross=False):\n        super().__init__()\n        self.heads, self.kv_heads = c.heads, c.kv_heads\n        self.hd = c.dim // c.heads\n        self.q = nn.Linear(c.dim, c.dim, bias=False)\n        self.k = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)\n        self.v = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)\n        self.out = nn.Linear(c.dim, c.dim, bias=False)\n        self.rotary = None if cross else Rotary(self.hd, c.max_positions)\n        self.causal, self.cross = causal, cross\n        self.window = c.local_window if causal and not cross else 0\n        self.dropout = c.dropout\n\n    def forward(self, x, memory=None, past=None, position=0, use_cache=False):\n        b, t, _ = x.shape\n        q = self.q(x).view(b, t, self.heads, self.hd).transpose(1, 2)\n        if self.cross and past is not None:\n            k, v = past  # Encoder K/V projections are also cached during decoding.\n        else:\n            source = x if memory is None else memory\n            s = source.size(1)\n            k = self.k(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)\n            v = self.v(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)\n            if self.rotary is not None:\n                k = self.rotary(k, position)\n            if past is not None:\n                k, v = torch.cat((past[0], k), -2), torch.cat((past[1], v), -2)\n        if self.rotary is not None:\n            q = self.rotary(q, position)\n        new_cache = (k, v) if use_cache else None\n        # Portable GQA: project/cache fewer KV heads, repeat only for the SDPA call.\n        # This is not a block-sparse implementation or a guaranteed training speedup.\n        if self.kv_heads != self.heads:\n            repeats = self.heads // self.kv_heads\n            k = k.repeat_interleave(repeats, dim=1)\n            v = v.repeat_interleave(repeats, dim=1)\n        mask, is_causal = None, False\n        if self.causal:\n            s = k.size(-2)\n            if self.window or (past is not None and t > 1):\n                qi = torch.arange(s - t, s, device=x.device)[:, None]\n                ki = torch.arange(s, device=x.device)[None, :]\n                mask = ki <= qi\n                if self.window:\n                    mask = mask & ((qi - ki) < self.window)\n            elif past is None:\n                is_causal = True\n            # Cached single-token query must see all prior keys. is_causal=True\n            # here would incorrectly use the upper-left non-square mask.\n        y = F.scaled_dot_product_attention(\n            q, k, v, attn_mask=mask, is_causal=is_causal,\n            dropout_p=self.dropout if self.training else 0.0,\n        )\n        y = self.out(y.transpose(1, 2).contiguous().view(b, t, -1))\n        return y, new_cache\n\n\nclass SwiGLU(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.gate = nn.Linear(c.dim, c.ff_dim, bias=False)\n        self.up = nn.Linear(c.dim, c.ff_dim, bias=False)\n        self.down = nn.Linear(c.ff_dim, c.dim, bias=False)\n\n    def forward(self, x):\n        return self.down(F.silu(self.gate(x)) * self.up(x))\n\n\nclass Block(nn.Module):\n    def __init__(self, c, encoder=False):\n        super().__init__()\n        self.attn_norm, self.ff_norm = RMSNorm(c.dim), RMSNorm(c.dim)\n        self.attn = Attention(c, causal=not encoder)\n        self.ff = SwiGLU(c)\n        self.drop = nn.Dropout(c.dropout)\n\n    def forward(self, x, past=None, position=0, use_cache=False):\n        y, cache = self.attn(self.attn_norm(x), past=past, position=position, use_cache=use_cache)\n        x = x + self.drop(y)\n        x = x + self.drop(self.ff(self.ff_norm(x)))\n        return (x, cache) if use_cache else x\n\n\nclass CrossBlock(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.self_norm, self.cross_norm, self.ff_norm = (RMSNorm(c.dim) for _ in range(3))\n        self.self_attn = Attention(c)\n        self.cross_attn = Attention(c, causal=False, cross=True)\n        self.ff, self.drop = SwiGLU(c), nn.Dropout(c.dropout)\n\n    def forward(self, x, memory, past=None, cross_past=None, position=0, use_cache=False):\n        y, cache = self.self_attn(self.self_norm(x), past=past, position=position, use_cache=use_cache)\n        x = x + self.drop(y)\n        y, cross_cache = self.cross_attn(self.cross_norm(x), memory=memory, past=cross_past, use_cache=use_cache)\n        x = x + self.drop(y)\n        x = x + self.drop(self.ff(self.ff_norm(x)))\n        return (x, cache, cross_cache) if use_cache else x\n\n\nclass LegalLM(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.config = c\n        self.embedding = nn.Embedding(c.vocab_size, c.dim)\n        # Output projection uses embedding.weight; there is no second vocabulary matrix.\n        if c.architecture == "encoder_decoder":\n            self.encoder = nn.ModuleList(Block(c, encoder=True) for _ in range(c.encoder_layers))\n            self.encoder_norm = RMSNorm(c.dim)\n            self.blocks = nn.ModuleList(CrossBlock(c) for _ in range(c.layers))\n        else:\n            self.blocks = nn.ModuleList(Block(c) for _ in range(c.layers))\n        if c.rnn_layers:\n            self.rnn = nn.LSTM(c.dim, c.dim, num_layers=c.rnn_layers, batch_first=True)\n            self.rnn_norm = RMSNorm(c.dim)\n        self.final_norm = RMSNorm(c.dim)\n        self.apply(self._init)\n        # Scale residual projections to keep initial residual variance controlled.\n        std = 0.02 / math.sqrt(2 * c.layers)\n        for name, p in self.named_parameters():\n            if name.endswith(("attn.out.weight", "self_attn.out.weight", "cross_attn.out.weight", "ff.down.weight")):\n                nn.init.normal_(p, mean=0.0, std=std)\n\n    @staticmethod\n    def _init(m):\n        if isinstance(m, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(m.weight, mean=0.0, std=0.02)\n        elif isinstance(m, nn.LSTM):\n            for name, p in m.named_parameters():\n                if "weight" in name:\n                    nn.init.normal_(p, mean=0.0, std=0.02)\n                else:\n                    nn.init.zeros_(p)\n\n    def encode(self, source_ids):\n        x = self.embedding(source_ids)\n        for block in self.encoder:\n            if self.config.gradient_checkpointing and self.training:\n                x = checkpoint(block, x, use_reentrant=False)\n            else:\n                x = block(x)\n        return self.encoder_norm(x)\n\n    def forward(self, input_ids, source_ids=None, state=None, use_cache=False):\n        state = state or {}\n        position = state.get("position", 0)\n        x = self.embedding(input_ids)\n        new_state = {"position": position + input_ids.size(1), "self": []}\n        if self.config.rnn_layers:\n            y, recurrent_state = self.rnn(x, state.get("rnn"))\n            x = x + self.rnn_norm(y)\n            new_state["rnn"] = recurrent_state\n        if self.config.architecture == "encoder_decoder":\n            memory = state.get("memory")\n            if memory is None:\n                if source_ids is None:\n                    raise ValueError("encoder_decoder requires source_ids on the first call.")\n                memory = self.encode(source_ids)\n            new_state.update(memory=memory, cross=[])\n            for i, block in enumerate(self.blocks):\n                if use_cache:\n                    p = state.get("self", [None] * len(self.blocks))[i]\n                    cp = state.get("cross", [None] * len(self.blocks))[i]\n                    x, cache, cc = block(x, memory, p, cp, position, True)\n                    new_state["self"].append(cache)\n                    new_state["cross"].append(cc)\n                elif self.config.gradient_checkpointing and self.training:\n                    x = checkpoint(block, x, memory, use_reentrant=False)\n                else:\n                    x = block(x, memory)\n        else:\n            for i, block in enumerate(self.blocks):\n                if use_cache:\n                    p = state.get("self", [None] * len(self.blocks))[i]\n                    x, cache = block(x, p, position, True)\n                    new_state["self"].append(cache)\n                elif self.config.gradient_checkpointing and self.training:\n                    x = checkpoint(block, x, use_reentrant=False)\n                else:\n                    x = block(x)\n        x = self.final_norm(x)\n        return (x, new_state) if use_cache else x\n\n    def project(self, hidden):\n        return F.linear(hidden, self.embedding.weight)\n\n\ndef parameter_count(model):\n    # PyTorch de-duplicates shared Parameter objects here.\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\n\ndef parameter_formula(c):\n    d, h, kv = c.dim, c.dim // c.heads, c.kv_heads\n    attention = 2 * d * d + 2 * d * kv * h\n    ff = 3 * d * c.ff_dim\n    total = c.vocab_size * d + d\n    if c.architecture == "encoder_decoder":\n        total += c.encoder_layers * (attention + ff + 2 * d) + d\n        total += c.layers * (2 * attention + ff + 3 * d)\n    else:\n        total += c.layers * (attention + ff + 2 * d)\n    if c.rnn_layers:\n        total += c.rnn_layers * (8 * d * d + 8 * d) + d\n    return total\n\n\ndef make_batch(blocks, architecture, prefix_fraction=0.80, objective="full"):\n    # blocks has T+1 tokens. The target suffix is identical across architectures.\n    t = blocks.size(1) - 1\n    split = max(1, min(t, int(t * prefix_fraction)))\n    if architecture == "encoder_decoder":\n        source = blocks[:, :split]\n        inputs, targets = blocks[:, split - 1:-1], blocks[:, split:]\n    else:\n        source = None\n        inputs, targets = blocks[:, :-1], blocks[:, 1:].clone()\n        if objective == "continuation":\n            targets[:, :split - 1] = -100\n    return inputs, targets, source\n\n\ndef language_model_loss(hidden, targets, weight, chunk_tokens=256, checkpoint_chunks=False):\n    """Sum/token count lets validation aggregate the true token-weighted NLL."""\n    hidden, targets = hidden.reshape(-1, hidden.size(-1)), targets.reshape(-1)\n    total = hidden.new_zeros((), dtype=torch.float32)\n    count = (targets != -100).sum()\n    def loss_chunk(h, y, w):\n        # Keep logits in autocast precision until the bounded FP32 CE conversion.\n        return F.cross_entropy(F.linear(h, w).float(), y, ignore_index=-100, reduction="sum")\n    for i in range(0, targets.numel(), chunk_tokens):\n        h, y = hidden[i:i + chunk_tokens], targets[i:i + chunk_tokens]\n        if checkpoint_chunks and torch.is_grad_enabled():\n            total = total + checkpoint(loss_chunk, h, y, weight, use_reentrant=False)\n        else:\n            total = total + loss_chunk(h, y, weight)\n    return total / count.clamp_min(1), total.detach(), count.detach()\n\n\nif __name__ == "__main__":\n    for name in ("decoder", "encoder_decoder", "local_gqa", "hybrid"):\n        c = architecture_config(name)\n        with torch.device("meta"):\n            model = LegalLM(c)\n        n = parameter_count(model)\n        assert n == parameter_formula(c) and 50_000_000 <= n <= 300_000_000\n        print(name, f"{n:,}", asdict(c))\n'
(CODE/'legal_models.py').write_text(SOURCE,encoding="utf-8")
compile(SOURCE,'legal_models.py',"exec")
print("Wrote",'legal_models.py')

In [ ]:
SOURCE = '"""Phase-5 schemas, leakage protection, all-token cache, sparse-label metrics and calibration."""\nimport hashlib,json,math,os,shutil,zipfile\n\nfrom pathlib import Path\n\nimport numpy as np\n\nimport torch\n\nfrom torch.utils.data import Dataset\n\nfrom tokenizers import Tokenizer\n\nfrom sklearn.metrics import f1_score,precision_score,recall_score,confusion_matrix\n\ndef digest(value):return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False,separators=(\',\',\':\')).encode()).hexdigest()\n\ndef file_hash(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as handle:\n        for chunk in iter(lambda:handle.read(1<<20),b\'\'):h.update(chunk)\n    return h.hexdigest()\n\ndef read_json(path):\n    def unique(pairs):\n        result={}\n        for key,value in pairs:\n            if key in result:raise ValueError(f\'Duplicate JSON key: {key}\')\n            result[key]=value\n        return result\n    return json.loads(Path(path).read_text(encoding=\'utf-8-sig\'),object_pairs_hook=unique)\n\ndef atomic_json(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+\'.tmp\');tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding=\'utf-8\');tmp.replace(path)\n\ndef atomic_torch(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+\'.tmp\');torch.save(value,tmp);tmp.replace(path)\n\ndef alias(row,names):\n    keys=[name for name in names if name in row]\n    if len(keys)>1 and any(row[k]!=row[keys[0]] for k in keys[1:]):raise ValueError(f\'Conflicting fields: {keys}\')\n    return row[keys[0]] if keys else None\n\ndef shape(value):\n    if isinstance(value,dict):return {str(k):type(v).__name__ for k,v in list(value.items())[:8]}\n    return {\'type\':type(value).__name__,\'length\':len(value) if isinstance(value,list) else None}\n\ndef to_device(batch,device):return {k:v.to(device,non_blocking=True) if isinstance(v,torch.Tensor) else v for k,v in batch.items()}\n\nimport re,bisect\nDATA_FILES=[\'Train/train.json\',\'Train/dev.json\',\'Test/test.json\']\n\ndef records(value,split,audit=None):\n    if isinstance(value,dict) and set(value) in ({\'data\'},{\'records\'},{\'documents\'}):value=next(iter(value.values()))\n    if isinstance(value,list):rows=[(None,row) for row in value]\n    elif isinstance(value,dict):\n        texts=alias(value,(\'text\',\'sentences\',\'document\'));ids=alias(value,(\'id\',\'ids\',\'case_id\',\'doc_id\'))\n        labels=None if split==\'test\' else alias(value,(\'labels\',\'label\',\'gold\',\'statutes\',\'statute_ids\'))\n        if isinstance(ids,str) and texts is not None:rows=[(None,{\'id\':ids,\'text\':texts,\'labels\':labels})]\n        elif isinstance(ids,list) and isinstance(texts,list):\n            if len(ids)!=len(texts) or (labels is not None and (not isinstance(labels,list) or len(labels)!=len(ids))):raise ValueError(f\'{split}: column lengths differ.\')\n            rows=[(None,{\'id\':did,\'text\':texts[i],\'labels\':None if labels is None else labels[i]}) for i,did in enumerate(ids)]\n        elif isinstance(texts,dict):\n            for column in (ids,labels):\n                if column is not None and (not isinstance(column,dict) or set(column)!=set(texts)):raise ValueError(\'Column keys differ.\')\n            rows=[(None,{\'id\':key if ids is None else ids[key],\'text\':body,\'labels\':None if labels is None else labels[key]}) for key,body in texts.items()]\n        else:rows=list(value.items())\n    else:raise ValueError(f\'{split}: unsupported input structure: {shape(value)}\')\n    grouped={};exact=set();copies=0;different=0\n    for key,row in rows:\n        if isinstance(row,str) and key is not None:row={\'id\':key,\'text\':row}\n        if not isinstance(row,dict):raise ValueError(f\'{split}: document must be an object.\')\n        did=alias(row,(\'id\',\'case_id\',\'doc_id\'));did=key if did is None else did\n        if not isinstance(did,str) or not did.strip():raise ValueError(f\'{split}: original nonempty string IDs are required.\')\n        if key is not None and key!=did:raise ValueError(\'Mapping ID and explicit case ID differ.\')\n        text=alias(row,(\'text\',\'sentences\',\'document\'))\n        if isinstance(text,list) and all(isinstance(x,str) for x in text):text=\'\\n\'.join(text)\n        if not isinstance(text,str):raise ValueError(f\'{did}: expected text or an ordered sentence-string list.\')\n        gold=None if split==\'test\' else alias(row,(\'labels\',\'label\',\'gold\',\'statutes\',\'statute_ids\'))\n        if split!=\'test\':\n            if not isinstance(gold,list) or any(type(x) is not int or x<0 for x in gold):raise ValueError(f\'{did}: labels must be a list of nonnegative integer statute IDs.\')\n            gold=sorted(set(gold))\n        signature=(did,digest(text),None if gold is None else tuple(gold))\n        if signature in exact:copies+=1;continue\n        exact.add(signature)\n        entry=grouped.setdefault(did,{\'texts\':[],\'text_hashes\':set(),\'gold\':set(),\'rows\':0,\'label_sets\':set()})\n        entry[\'rows\']+=1;sha=digest(text)\n        if sha not in entry[\'text_hashes\']:entry[\'text_hashes\'].add(sha);entry[\'texts\'].append(text)\n        if gold is not None:entry[\'gold\'].update(gold);entry[\'label_sets\'].add(tuple(gold))\n    result=[{\'id\':did,\'source_id\':did,\'text\':\'\\n\\n\'.join(v[\'texts\']),\n             \'gold\':None if split==\'test\' else sorted(v[\'gold\']),\'split\':split} for did,v in grouped.items()]\n    if not result:raise ValueError(f\'{split}: empty split.\')\n    if audit is not None:audit.update(raw_rows=len(rows),case_count=len(result),exact_copies_removed=copies,\n        repeated_cases_combined=sum(v[\'rows\']>1 for v in grouped.values()),\n        cases_with_different_label_sets=sum(len(v[\'label_sets\'])>1 for v in grouped.values()),\n        policy=\'One original ID per case; distinct texts concatenate in source order; supplied train/dev label sets union. Test labels are discarded.\')\n    return result\n\ndef load_records(root,return_audit=False):\n    root=Path(root);audit={};all_rows={}\n    for split,name in zip((\'train\',\'dev\',\'test\'),DATA_FILES):\n        audit[split]={};all_rows[split]=records(read_json(root/name),split,audit[split])\n    test_ids={r[\'id\'] for r in all_rows[\'test\']};dev_ids={r[\'id\'] for r in all_rows[\'dev\']}\n    audit[\'dev_test_source_overlap_removed\']=sorted(dev_ids&test_ids)\n    all_rows[\'dev\']=[r for r in all_rows[\'dev\'] if r[\'id\'] not in test_ids]\n    held_ids=dev_ids|test_ids\n    held_text={digest(\' \'.join(r[\'text\'].split())) for s in (\'dev\',\'test\') for r in all_rows[s]}\n    removed=[r[\'id\'] for r in all_rows[\'train\'] if r[\'id\'] in held_ids or digest(\' \'.join(r[\'text\'].split())) in held_text]\n    remove=set(removed);all_rows[\'train\']=[r for r in all_rows[\'train\'] if r[\'id\'] not in remove]\n    audit[\'training_held_out_cases_or_text_copies_removed\']=removed\n    if any(not all_rows[s] for s in (\'train\',\'dev\',\'test\')):raise ValueError(\'Leakage filtering left an empty split; inspect IDs and schema before training.\')\n    result=sum([all_rows[s] for s in (\'train\',\'dev\',\'test\')],[])\n    if len({r[\'id\'] for r in result})!=len(result):raise ValueError(\'IDs overlap after case filtering.\')\n    return (result,audit) if return_audit else result\n\ndef prepare_data(root,tokenizer_path,output,chunk_tokens=384,overlap=48):\n    root=Path(root);output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    if not 0<=overlap<chunk_tokens:raise ValueError(\'Require 0 <= overlap < chunk_tokens.\')\n    identity=digest({\'files\':{name:file_hash(root/name) for name in DATA_FILES},\'tokenizer\':file_hash(tokenizer_path),\n        \'chunk_tokens\':chunk_tokens,\'overlap\':overlap,\'version\':1})\n    manifest=output/\'manifest.json\'\n    if manifest.exists():\n        old=read_json(manifest)\n        if old[\'fingerprint\']!=identity:raise ValueError(\'Input/cache changed. Use a new PROJECT.\')\n        if any(old[\'docs\'][i][\'gold\'] is not None for i in old[\'splits\'][\'test\']):raise ValueError(\'Unsafe cache contains test targets.\')\n        if (output/\'tokens.i32\').stat().st_size!=4*old[\'total_tokens\']:raise ValueError(\'Incomplete token cache.\')\n        return str(manifest)\n    rows,audit=load_records(root,True)\n    label_ids=sorted({y for r in rows if r[\'split\']!=\'test\' for y in r[\'gold\']})\n    if not label_ids:raise ValueError(\'No public training/development labels.\')\n    label_count=max(label_ids)+1\n    if label_count>50000:raise ValueError(\'Label dimension exceeds 50000. Verify that statutory sections are already mapped to evaluator class IDs.\')\n    tokenizer=Tokenizer.from_file(str(tokenizer_path));tokenizer.no_padding();tokenizer.no_truncation()\n    unk=tokenizer.token_to_id(\'<unk>\');unk=0 if unk is None else unk\n    docs=[];total=chunks=empty=0;temporary=output/\'tokens.i32.tmp\'\n    cue=re.compile(r\'\\b(?:sections?|articles?|acts?|statutes?|codes?|provisions?|ipc|crpc|constitution|rules?)\\b\',re.I)\n    with temporary.open(\'wb\') as handle:\n        for index,row in enumerate(rows):\n            encoded=tokenizer.encode(row[\'text\'],add_special_tokens=False);ids=encoded.ids;offsets=encoded.offsets\n            empty+=int(not ids);ids=ids or [unk];offsets=offsets or [(0,0)]\n            np.asarray(ids,dtype=\'<i4\').tofile(handle);n=len(ids);final=max(0,n-chunk_tokens)\n            starts=list(range(0,final+1,chunk_tokens-overlap))\n            if starts[-1]!=final:starts.append(final)\n            windows=[];scores=[]\n            for start in starts:\n                length=min(chunk_tokens,n-start);windows.append([total+start,length,start/max(1,n-1)])\n                left=offsets[start][0];right=offsets[start+length-1][1]\n                scores.append(len(cue.findall(row[\'text\'][left:right])))\n            order=list(dict.fromkeys([0,len(windows)-1]+sorted(range(len(windows)),key=lambda j:(-scores[j],j))))\n            docs.append({\'id\':row[\'id\'],\'source_id\':row[\'id\'],\'gold\':row[\'gold\'],\'split\':row[\'split\'],\n                \'token_count\':n,\'chunks\':windows,\'evidence_order\':order,\'feature_start\':chunks,\'text_sha256\':digest(row[\'text\'])})\n            total+=n;chunks+=len(windows)\n            if (index+1)%2000==0:print(\'Tokenized cases:\',index+1,flush=True)\n    temporary.replace(output/\'tokens.i32\')\n    splits={s:[i for i,d in enumerate(docs) if d[\'split\']==s] for s in (\'train\',\'dev\',\'test\')}\n    splits[\'all\']=splits[\'train\']+splits[\'dev\'];splits[\'every\']=list(range(len(docs)))\n    support=np.zeros(label_count,dtype=np.int64)\n    for i in splits[\'train\']:support[docs[i][\'gold\']]+=1\n    value={\'fingerprint\':identity,\'docs\':docs,\'splits\':splits,\'total_tokens\':total,\'total_chunks\':chunks,\n        \'chunk_tokens\':chunk_tokens,\'overlap\':overlap,\'label_ids\':label_ids,\'label_count\':label_count,\'train_support\':support.tolist(),\n        \'tokenizer_sha256\':file_hash(tokenizer_path),\'audit\':{**audit,\'empty_documents\':empty,\'all_tokens_retained\':True,\n        \'case_counts\':{s:len(v) for s,v in splits.items()},\'label_count_including_id_gaps\':label_count,\n        \'labels_without_training_positives\':np.flatnonzero(support==0).tolist(),\n        \'adaptation_view\':\'First, last, then statute-cue windows; full branch retains all windows.\',\n        \'full_feature_gib_per_512d_parent\':chunks*1024*2/1024**3}}\n    atomic_json(manifest,value);return str(manifest)\n\ndef target_matrix(docs,label_count):\n    y=np.zeros((len(docs),label_count),dtype=np.float32)\n    for i,doc in enumerate(docs):\n        if doc[\'gold\'] is None:raise ValueError(\'Test targets cannot be used for fitting or selection.\')\n        y[i,doc[\'gold\']]=1\n    return y\n\nclass Bags(Dataset):\n    def __init__(self,manifest,split,features=None,teacher=None,view=\'all\',adapt_chunks=3):\n        self.path=Path(manifest);self.data=read_json(self.path);self.rows=self.data[\'splits\'][split]\n        self.tokens=np.memmap(self.path.parent/\'tokens.i32\',dtype=\'<i4\',mode=\'r\');self.features=None\n        if features:\n            info=read_json(Path(features)/\'manifest.json\')\n            if not info[\'complete\'] or info[\'data_fingerprint\']!=self.data[\'fingerprint\']:raise ValueError(\'Feature cache mismatch/incomplete.\')\n            self.features=np.memmap(Path(features)/\'features.f16\',dtype=\'<f2\',mode=\'r\',shape=tuple(info[\'shape\']))\n        if view not in (\'all\',\'evidence\'):raise ValueError(\'Unknown context view.\')\n        self.teacher,self.view,self.adapt_chunks=teacher,view,adapt_chunks\n    def __len__(self):return len(self.rows)\n    def __getitem__(self,index):return self.document(self.rows[index])\n    def document(self,index):\n        doc=self.data[\'docs\'][index];windows=doc[\'chunks\'];keep=list(range(len(windows)))\n        if self.view==\'evidence\':keep=sorted(doc[\'evidence_order\'][:self.adapt_chunks])\n        position=[[windows[j][2],1-windows[j][2],windows[j][2]**2,float(j==0),float(j==len(windows)-1),math.log1p(windows[j][1])/8] for j in keep]\n        target=None\n        if doc[\'gold\'] is not None:\n            target=np.zeros(self.data[\'label_count\'],dtype=np.float32);target[doc[\'gold\']]=1\n        return {\'index\':index,\'id\':doc[\'id\'],\'chunks\':[np.array(self.tokens[windows[j][0]:windows[j][0]+windows[j][1]],dtype=np.int64) for j in keep] if self.features is None else None,\n            \'features\':torch.from_numpy(np.array(self.features[[doc[\'feature_start\']+j for j in keep]],dtype=np.float32)) if self.features is not None else None,\n            \'position\':position,\'target\':target,\'teacher\':self.teacher[doc[\'id\']] if self.teacher is not None else None}\n\ndef collate(rows):\n    B=len(rows);S=max(len(r[\'position\']) for r in rows);positions=torch.zeros(B,S,6);mask=torch.zeros(B,S,dtype=torch.bool)\n    owners=[];chunks=[];features=None\n    if rows[0][\'features\'] is not None:features=torch.zeros(B,S,rows[0][\'features\'].shape[-1])\n    for b,row in enumerate(rows):\n        n=len(row[\'position\']);positions[b,:n]=torch.tensor(row[\'position\']);mask[b,:n]=True\n        if features is not None:features[b,:n]=row[\'features\']\n        else:chunks.extend(row[\'chunks\']);owners.extend(b*S+j for j in range(n))\n    targets=None if rows[0][\'target\'] is None else torch.from_numpy(np.stack([r[\'target\'] for r in rows]))\n    return {\'chunks\':chunks,\'owners\':owners,\'features\':features,\'position\':positions,\'mask\':mask,\'target\':targets,\n        \'teacher\':torch.stack([torch.as_tensor(r[\'teacher\']) for r in rows]) if rows[0][\'teacher\'] is not None else None,\'rows\':rows}\n\ndef probabilities(logits):\n    x=np.asarray(logits,dtype=np.float64);return 1/(1+np.exp(-np.clip(x,-40,40)))\n\ndef eval_dimension(y):\n    active=np.flatnonzero(np.asarray(y).sum(0));return int(active.max()+1) if len(active) else np.asarray(y).shape[1]\n\ndef metric(y,probability,threshold=.5):\n    y=np.asarray(y,dtype=np.uint8);p=np.asarray(probability);pred=p>=np.asarray(threshold)\n    n=eval_dimension(y);y=y[:,:n];pred=pred[:,:n]\n    return {\'macro_f1\':float(f1_score(y,pred,average=\'macro\',zero_division=0)),\n        \'micro_f1\':float(f1_score(y,pred,average=\'micro\',zero_division=0)),\n        \'subset_accuracy\':float(np.all(y==pred,axis=1).mean()),\'label_dimension_from_truth\':n}\n\ndef fit_thresholds(y,p,shrink=20.):\n    y=np.asarray(y);p=np.asarray(p);grid=np.array([.03,.05,.075,.1,.15,.2,.25,.3,.35,.4,.45,.5,.6,.7,.8])\n    global_t=max(grid,key=lambda t:metric(y,p,t)[\'macro_f1\']);threshold=np.full(y.shape[1],global_t,dtype=float)\n    support=y.sum(0)\n    for j in range(y.shape[1]):\n        if support[j]<3:continue\n        pred=p[:,j,None]>=grid[None,:];tp=(pred*y[:,j,None]).sum(0);denom=pred.sum(0)+support[j]\n        f1=np.divide(2*tp,denom,out=np.zeros(len(grid)),where=denom>0)\n        tied=np.flatnonzero(f1>=f1.max()-1e-12);best=tied[np.argmin(abs(grid[tied]-global_t))]\n        strength=support[j]/(support[j]+shrink);threshold[j]=(1-strength)*global_t+strength*grid[best]\n    if metric(y,p,threshold)[\'macro_f1\']<metric(y,p,global_t)[\'macro_f1\']:threshold[:]=global_t\n    return threshold.tolist()\n\ndef predict_labels(probability,threshold,allowed_ids):\n    p=np.asarray(probability);binary=p>=np.asarray(threshold);allowed=set(allowed_ids)\n    return [[int(j) for j in np.flatnonzero(row) if j in allowed] for row in binary]\n\ndef write_submission(ids,preds,path):\n    if len(ids)!=len(set(ids)) or len(ids)!=len(preds):raise ValueError(\'Submission IDs/count mismatch.\')\n    if any(not isinstance(row,list) or any(type(y) is not int or y<0 for y in row) or len(row)!=len(set(row)) for row in preds):raise ValueError(\'Predictions require unique nonnegative integer label IDs.\')\n    rows=[{\'id\':did,\'pred\':sorted(p)} for did,p in zip(ids,preds)];raw=json.dumps(rows,ensure_ascii=False).encode()\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_suffix(\'.tmp\')\n    with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED) as archive:archive.writestr(\'prediction.json\',raw)\n    tmp.replace(path);return rows,hashlib.sha256(raw).hexdigest()\n'
(CODE/'lsi_data.py').write_text(SOURCE,encoding="utf-8")
compile(SOURCE,'lsi_data.py',"exec")
print("Wrote",'lsi_data.py')

In [ ]:
SOURCE = '"""Full documents, train-only sparse vocabulary and test-target isolation."""\nimport gc, math, shutil, json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom tokenizers import Tokenizer\nfrom scipy import sparse\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nimport joblib\nfrom lsi_data import load_records, read_json, atomic_json, file_hash, digest, DATA_FILES\n\ndef prepare(root, tokenizer_path, output, cfg):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    fingerprint=digest({\'sources\':{n:file_hash(Path(root)/n) for n in DATA_FILES},\n        \'tokenizer\':file_hash(tokenizer_path),\'chunk\':cfg[\'chunk_tokens\'],\n        \'word_features\':cfg[\'word_features\'],\'char_features\':cfg[\'char_features\'],\'version\':3})\n    manifest=output/\'manifest.json\'\n    if manifest.exists():\n        old=read_json(manifest)\n        if old[\'fingerprint\']!=fingerprint:raise ValueError(\'Inputs/settings changed. Use a new PROJECT.\')\n        for name,size in old[\'byte_sizes\'].items():\n            if not (output/name).is_file() or (output/name).stat().st_size!=size:raise ValueError(\'Incomplete cache: \'+name)\n        if any(d[\'gold\'] is not None for d in old[\'docs\'] if d[\'split\']==\'test\'):raise ValueError(\'Unsafe test cache.\')\n        return str(manifest)\n    rows,audit=load_records(root,True)\n    splits={s:[i for i,r in enumerate(rows) if r[\'split\']==s] for s in (\'train\',\'dev\',\'test\')}\n    label_ids=sorted({j for r in rows if r[\'split\']!=\'test\' for j in r[\'gold\']})\n    C=max(label_ids)+1\n    if C>5000:raise ValueError(\'Unexpected label count; verify mapped statute IDs.\')\n    support=np.zeros(C,dtype=np.int64)\n    for i in splits[\'train\']:support[rows[i][\'gold\']]+=1\n    tok=Tokenizer.from_file(str(tokenizer_path));tok.no_padding();tok.no_truncation()\n    unknown=tok.token_to_id(\'<unk>\');unknown=0 if unknown is None else unknown\n    docs=[];offset=0\n    with (output/\'tokens.i32\').open(\'wb\') as handle:\n        for i,r in enumerate(rows):\n            ids=tok.encode(r[\'text\'],add_special_tokens=False).ids or [unknown]\n            np.asarray(ids,dtype=\'<i4\').tofile(handle)\n            # Non-overlapping chunks cover each token exactly once, including the final partial chunk.\n            windows=[[offset+a,min(cfg[\'chunk_tokens\'],len(ids)-a)] for a in range(0,len(ids),cfg[\'chunk_tokens\'])]\n            docs.append({\'id\':r[\'id\'],\'gold\':r[\'gold\'],\'split\':r[\'split\'],\'chunks\':windows,\'token_count\':len(ids)})\n            offset+=len(ids)\n            if (i+1)%5000==0:print(\'Tokenized\',i+1,\'cases\',flush=True)\n    train_text=[rows[i][\'text\'] for i in splits[\'train\']]\n    vectorizers=[];blocks=[]\n    for kind,limit in [(\'word\',cfg[\'word_features\']),(\'char\',cfg[\'char_features\'])]:\n        if not limit:continue\n        vector=TfidfVectorizer(analyzer=\'word\' if kind==\'word\' else \'char_wb\',\n            ngram_range=(1,2) if kind==\'word\' else (3,5),max_features=limit,\n            min_df=min(3 if kind==\'word\' else 8,max(1,len(train_text)//4)),\n            sublinear_tf=True,strip_accents=\'unicode\',lowercase=True,dtype=np.float32)\n        vector.fit(train_text) # No dev/test vocabulary or IDF fitting.\n        print(\'Sparse vocabulary:\',kind,len(vector.vocabulary_),flush=True)\n        pieces=[vector.transform([r[\'text\'] for r in rows[a:a+512]]) for a in range(0,len(rows),512)]\n        blocks.append(sparse.vstack(pieces,format=\'csr\'));vectorizers.append(vector);del pieces;gc.collect()\n    matrix=sparse.hstack(blocks,format=\'csr\',dtype=np.float32)/math.sqrt(len(blocks));matrix=matrix.tocsr()\n    matrix.sort_indices();del blocks,train_text;gc.collect()\n    np.save(output/\'sparse_indices.npy\',matrix.indices.astype(np.int64))\n    np.save(output/\'sparse_indptr.npy\',matrix.indptr.astype(np.int64))\n    np.save(output/\'sparse_values.npy\',matrix.data.astype(np.float32))\n    joblib.dump(vectorizers,output/\'vectorizers.joblib\')\n    value={\'fingerprint\':fingerprint,\'tokenizer_sha256\':file_hash(tokenizer_path),\'docs\':docs,\'splits\':splits,\n        \'label_count\':C,\'label_ids\':label_ids,\'support\':support.tolist(),\'sparse_dim\':matrix.shape[1],\n        \'token_count\':offset,\'chunk_count\':sum(len(d[\'chunks\']) for d in docs),\'audit\':audit,\n        \'training_policy\':\'train only; full document; no test targets; no automatic train+dev refit\'}\n    names=[\'tokens.i32\',\'sparse_indices.npy\',\'sparse_indptr.npy\',\'sparse_values.npy\',\'vectorizers.joblib\']\n    value[\'byte_sizes\']={n:(output/n).stat().st_size for n in names};atomic_json(manifest,value)\n    print(\'Data counts:\',{s:len(v) for s,v in splits.items()},\'labels\',C,\'chunks\',value[\'chunk_count\'],flush=True)\n    return str(manifest)\n\nclass Documents:\n    def __init__(self,manifest):\n        self.path=Path(manifest);self.data=read_json(manifest)\n        self.tokens=np.memmap(self.path.parent/\'tokens.i32\',mode=\'r\',dtype=\'<i4\')\n        self.indices=np.load(self.path.parent/\'sparse_indices.npy\',mmap_mode=\'r\')\n        self.indptr=np.load(self.path.parent/\'sparse_indptr.npy\',mmap_mode=\'r\')\n        self.values=np.load(self.path.parent/\'sparse_values.npy\',mmap_mode=\'r\')\n    def batch(self,indices,device):\n        chunks=[];counts=[];features=[];values=[];offsets=[0];targets=[];positions=[]\n        for i in indices:\n            d=self.data[\'docs\'][int(i)];counts.append(len(d[\'chunks\']))\n            for j,(a,n) in enumerate(d[\'chunks\']):\n                chunks.append(np.array(self.tokens[a:a+n],dtype=np.int64))\n                positions.append([j/max(1,len(d[\'chunks\'])-1),float(j==0),float(j==len(d[\'chunks\'])-1)])\n            a,b=self.indptr[i:i+2];features.extend(self.indices[a:b]);values.extend(self.values[a:b]);offsets.append(len(features))\n            if d[\'gold\'] is not None:\n                y=np.zeros(self.data[\'label_count\'],np.float32);y[d[\'gold\']]=1;targets.append(y)\n        if targets and len(targets)!=len(indices):raise ValueError(\'Mixed train and test batches are forbidden.\')\n        return {\'indices\':list(map(int,indices)),\'chunks\':chunks,\'counts\':counts,\n            \'position\':torch.as_tensor(positions,dtype=torch.float32,device=device),\n            \'sparse_ids\':torch.as_tensor(features,dtype=torch.long,device=device),\n            \'sparse_values\':torch.as_tensor(values,dtype=torch.float32,device=device),\n            \'sparse_offsets\':torch.as_tensor(offsets,dtype=torch.long,device=device),\n            \'target\':torch.as_tensor(np.stack(targets),device=device) if targets else None}\n\ndef chunk_tensors(chunks,device):\n    T=max(map(len,chunks));ids=torch.zeros(len(chunks),T,dtype=torch.long,device=device)\n    valid=torch.zeros_like(ids,dtype=torch.bool)\n    for i,row in enumerate(chunks):\n        ids[i,:len(row)]=torch.as_tensor(row,device=device);valid[i,:len(row)]=True\n    return ids,valid\n\n'
(CODE/'s5_data.py').write_text(SOURCE,encoding="utf-8")
compile(SOURCE,'s5_data.py',"exec")
print("Wrote",'s5_data.py')

In [ ]:
SOURCE = '"""One jointly trained own-decoder model: token LWAN, document BiGRU, lexical residual."""\nimport math, contextlib\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom legal_models import LegalLM,ModelConfig\nfrom s5_data import chunk_tensors\n\nclass FullDocumentModel(nn.Module):\n    def __init__(self,foundation_config,cfg):\n        super().__init__();self.cfg=cfg;self.backbone=LegalLM(ModelConfig(**foundation_config))\n        if self.backbone.config.architecture!=\'decoder\':raise ValueError(\'This candidate uses only main_decoder.\')\n        self.backbone.config.gradient_checkpointing=False\n        # Deterministic chunk encoder is required for exact recomputation. Dropout remains in the document head.\n        for m in self.backbone.modules():\n            if isinstance(m,nn.Dropout):m.p=0.\n            if hasattr(m,\'dropout\') and isinstance(m.dropout,(int,float)):m.dropout=0.\n        d=self.backbone.config.dim;h=cfg[\'token_dim\'];g=cfg[\'document_dim\'];C=cfg[\'label_count\']\n        self.C,self.h,self.d=C,h,d\n        self.token_project=nn.Sequential(nn.LayerNorm(d),nn.Linear(d,h),nn.Tanh())\n        self.token_query=nn.Parameter(torch.empty(C,h));nn.init.normal_(self.token_query,std=.1)\n        self.token_output=nn.Parameter(torch.empty(C,h));nn.init.normal_(self.token_output,std=.02)\n        self.chunk_project=nn.Sequential(nn.LayerNorm(2*d+3),nn.Linear(2*d+3,g),nn.GELU())\n        self.context=nn.GRU(g,g//2,bidirectional=True,batch_first=True)\n        self.context_norm=nn.LayerNorm(g);self.dropout=nn.Dropout(cfg[\'dropout\'])\n        self.document_query=nn.Parameter(torch.empty(C,g));nn.init.normal_(self.document_query,std=.1)\n        self.document_output=nn.Parameter(torch.empty(C,g));nn.init.normal_(self.document_output,std=.02)\n        self.bias=nn.Parameter(torch.as_tensor(cfg[\'initial_bias\'],dtype=torch.float32))\n        self.lexical=nn.EmbeddingBag(cfg[\'sparse_dim\'],C,mode=\'sum\',include_last_offset=True)\n        nn.init.zeros_(self.lexical.weight)\n\n    def encode(self,ids,valid):\n        # Right-padded causal decoder: valid tokens never attend to padding that follows them.\n        hidden=self.backbone(ids)\n        value=self.token_project(hidden).float()\n        # Explicit FP32 scope: autocast can downcast einsum even when its inputs are float().\n        with torch.autocast(device_type=ids.device.type,enabled=False):\n            score=torch.einsum(\'nth,lh->ntl\',value.float(),self.token_query.float())/math.sqrt(self.h)\n            score=score.masked_fill(~valid[:,:,None],float(\'-inf\'))\n            logz=score.logsumexp(1)\n            weights=score.softmax(1);pooled=torch.einsum(\'ntl,nth->nlh\',weights,value.float())\n        mean=(hidden.float()*valid[:,:,None]).sum(1)/valid.sum(1)[:,None]\n        last=hidden[torch.arange(len(ids),device=ids.device),valid.sum(1)-1].float()\n        return torch.cat((pooled.flatten(1),logz,mean,last),1)\n\n    def classify(self,features,batch):\n        C,h=self.C,self.h\n        pooled=features[:,:C*h].reshape(-1,C,h);logz=features[:,C*h:C*h+C]\n        global_chunks=features[:,C*h+C:]\n        counts=batch[\'counts\'];S=max(counts);B=len(counts)\n        global_pad=features.new_zeros(B,S,2*self.d+3);mask=torch.zeros(B,S,dtype=torch.bool,device=features.device)\n        token_vectors=[];cursor=0\n        for b,n in enumerate(counts):\n            w=logz[cursor:cursor+n].softmax(0)\n            token_vectors.append((pooled[cursor:cursor+n]*w[:,:,None]).sum(0))\n            global_pad[b,:n]=torch.cat((global_chunks[cursor:cursor+n],batch[\'position\'][cursor:cursor+n]),1)\n            mask[b,:n]=True;cursor+=n\n        token_vectors=self.dropout(torch.stack(token_vectors))\n        token_logits=(token_vectors*self.token_output[None]).sum(-1)\n        x=self.chunk_project(global_pad)\n        packed=nn.utils.rnn.pack_padded_sequence(x,torch.as_tensor(counts),batch_first=True,enforce_sorted=False)\n        packed,_=self.context(packed)\n        x,_=nn.utils.rnn.pad_packed_sequence(packed,batch_first=True,total_length=S)\n        x=self.dropout(self.context_norm(x)).float()\n        with torch.autocast(device_type=features.device.type,enabled=False):\n            a=torch.einsum(\'bsh,lh->bsl\',x.float(),self.document_query.float())/math.sqrt(x.shape[-1])\n            a=a.masked_fill(~mask[:,:,None],float(\'-inf\')).softmax(1)\n            representation=torch.einsum(\'bsl,bsh->blh\',a,x.float())\n            neural=token_logits.float()+(representation*self.document_output[None].float()).sum(-1)+self.bias.float()\n        # EmbeddingBag implements a sparse linear residual, learned in the SAME loss/checkpoint.\n        # FP32 here avoids unsupported FP16 per_sample_weights combinations on older CUDA builds.\n        with torch.autocast(device_type=features.device.type,enabled=False):\n            values=F.dropout(batch[\'sparse_values\'],p=self.cfg.get(\'lexical_dropout\',.15),training=self.training)\n            lexical=self.lexical(batch[\'sparse_ids\'],batch[\'sparse_offsets\'],per_sample_weights=values)\n        return neural+lexical,neural\n\ndef amp(device):\n    return torch.autocast(\'cuda\',dtype=torch.float16) if device.type==\'cuda\' else contextlib.nullcontext()\n\ndef encode_all(model,batch,device,chunk_batch):\n    return torch.cat([model.encode(*chunk_tensors(batch[\'chunks\'][a:a+chunk_batch],device))\n        for a in range(0,len(batch[\'chunks\']),chunk_batch)],0)\n\ndef loss_fn(logits,neural,target,pos_weight,normalizer):\n    if target is None:raise ValueError(\'Test data cannot enter a fitting loss.\')\n    main=(F.binary_cross_entropy_with_logits(logits.float(),target,pos_weight=pos_weight,reduction=\'none\')/normalizer).mean()\n    auxiliary=(F.binary_cross_entropy_with_logits(neural.float(),target,pos_weight=pos_weight,reduction=\'none\')/normalizer).mean()\n    return main+.2*auxiliary\n\ndef cached_backward(model,batch,device,scaler,pos_weight,normalizer,divisor=1):\n    """Chain-rule VJP: exact document gradient, one chunk microbatch activation graph at a time."""\n    step=model.cfg[\'chunk_batch\']\n    with torch.no_grad(),amp(device):cached=encode_all(model,batch,device,step)\n    cached=cached.detach().requires_grad_(True)\n    with amp(device):\n        logits,neural=model.classify(cached,batch)\n        loss=loss_fn(logits,neural,batch[\'target\'],pos_weight,normalizer)\n    scaler.scale(loss/divisor).backward()\n    representation_gradient=cached.grad.detach()\n    # The representation gradient already contains GradScaler\'s scale. Do not scale it twice.\n    for a in range(0,len(batch[\'chunks\']),step):\n        with amp(device):value=model.encode(*chunk_tensors(batch[\'chunks\'][a:a+step],device))\n        torch.autograd.backward(value,representation_gradient[a:a+len(value)])\n    return float(loss.detach())\n\n'
(CODE/'s5_model.py').write_text(SOURCE,encoding="utf-8")
compile(SOURCE,'s5_model.py',"exec")
print("Wrote",'s5_model.py')

In [ ]:
SOURCE = '"""One-model Kaggle runtime: full-document gradient caching and synchronous two-GPU SGD."""\nimport argparse, contextlib, gc, json, math, os, random, shutil, time, zipfile\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nfrom s5_data import prepare, Documents\nfrom s5_model import FullDocumentModel, amp, encode_all, cached_backward\nfrom lsi_data import read_json, atomic_json, atomic_torch, digest, file_hash, probabilities, metric, write_submission\n\ndef setup():\n    rank=int(os.environ.get(\'RANK\',0));world=int(os.environ.get(\'WORLD_SIZE\',1));local=int(os.environ.get(\'LOCAL_RANK\',0))\n    device=torch.device(\'cuda\',local) if torch.cuda.is_available() else torch.device(\'cpu\')\n    if device.type==\'cuda\':torch.cuda.set_device(device)\n    if world>1:\n        if device.type==\'cuda\':dist.init_process_group(\'nccl\',device_id=device)\n        else:dist.init_process_group(\'gloo\')\n    torch.set_num_threads(1)\n    return rank,world,device\n\ndef barrier():\n    if dist.is_initialized():dist.barrier()\n\ndef build(cfg,checkpoint=None):\n    if checkpoint:\n        saved=torch.load(checkpoint,map_location=\'cpu\',weights_only=False)\n        if saved.get(\'training_origin\')!=\'own_phase1_phase5_full_document\':raise ValueError(\'Wrong model provenance.\')\n        model=FullDocumentModel(saved[\'foundation_config\'],cfg);model.load_state_dict(saved[\'model\'],strict=True)\n        return model,saved[\'provenance\']\n    parent=torch.load(cfg[\'phase1_checkpoint\'],map_location=\'cpu\',weights_only=False)\n    if parent.get(\'training_origin\')!=\'random_initialization_official_phase1_only\':raise ValueError(\'Use original Phase-1 weights.\')\n    if parent[\'tokenizer_sha256\']!=cfg[\'tokenizer_sha256\']:raise ValueError(\'Tokenizer and foundation do not match.\')\n    if parent[\'model_config\'][\'architecture\']!=\'decoder\':raise ValueError(\'Expected main_decoder/best_model.pt.\')\n    model=FullDocumentModel(parent[\'model_config\'],cfg);model.backbone.load_state_dict(parent[\'model\'],strict=True)\n    count=sum(p.numel() for p in model.backbone.parameters())\n    if not cfg.get(\'_synthetic_test\',False) and not 50_000_000<=count<=300_000_000:raise ValueError(\'Invalid foundation parameter count.\')\n    return model,{\'phase1_sha256\':file_hash(cfg[\'phase1_checkpoint\']),\'tokenizer_sha256\':cfg[\'tokenizer_sha256\'],\n        \'foundation_parameters\':count,\'original_phase1_only\':True}\n\ndef sync_gradients(model,world):\n    if world==1:return\n    # Explicit averaging is used because a logical document backward replays many encoder forwards.\n    # Do not wrap this model in DDP as well: doing so would average gradients twice.\n    bucket=[];size=0\n    def flush():\n        if not bucket:return\n        flat=torch.cat([p.grad.reshape(-1) for p in bucket]);dist.all_reduce(flat);flat.div_(world)\n        a=0\n        for p in bucket:\n            n=p.numel();p.grad.copy_(flat[a:a+n].view_as(p));a+=n\n        bucket.clear()\n    for p in model.parameters():\n        if not p.requires_grad:continue\n        if p.grad is None:p.grad=torch.zeros_like(p)\n        if size+p.numel()>4_000_000 and bucket:flush();size=0\n        bucket.append(p);size+=p.numel()\n    flush()\n\ndef rng_state(device):\n    return {\'python\':random.getstate(),\'numpy\':np.random.get_state(),\'torch\':torch.get_rng_state(),\n        \'cuda\':torch.cuda.get_rng_state(device) if device.type==\'cuda\' else None}\ndef restore_rng(state,device):\n    random.setstate(state[\'python\']);np.random.set_state(state[\'numpy\']);torch.set_rng_state(state[\'torch\'])\n    if device.type==\'cuda\':torch.cuda.set_rng_state(state[\'cuda\'],device)\n\ndef save(path,model,cfg,provenance,optimizer=None,scaler=None,cursor=None,state=None,rank=0,world=1,device=None):\n    random_states=None\n    if optimizer is not None:\n        current=rng_state(device);random_states=[None]*world\n        if world>1:dist.all_gather_object(random_states,current)\n        else:random_states=[current]\n    if rank==0:\n        result={\'training_origin\':\'own_phase1_phase5_full_document\',\'config\':cfg,\'foundation_config\':vars(model.backbone.config),\n            \'provenance\':provenance,\'model\':model.state_dict()}\n        if optimizer is not None:result.update(optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),cursor=cursor,state=state,rng=random_states)\n        atomic_torch(path,result)\n    barrier()\n\ndef epoch_batches(data,cfg,epoch,rank,world):\n    rows=list(data[\'splits\'][\'train\']);random.Random(cfg[\'seed\']+epoch).shuffle(rows)\n    # Local pools reduce padding in the document head. Chunks themselves are microbatched independently.\n    for a in range(0,len(rows),128):rows[a:a+128]=sorted(rows[a:a+128],key=lambda i:len(data[\'docs\'][i][\'chunks\']))\n    quantum=world*cfg[\'batch_docs\'];padding=(-len(rows))%quantum\n    rows+=([rows[i%len(rows)] for i in range(padding)])\n    local=rows[rank::world]\n    return [local[a:a+cfg[\'batch_docs\']] for a in range(0,len(local),cfg[\'batch_docs\'])]\n\n@torch.no_grad()\ndef predict(model,ds,split,cfg,rank,world,device,out):\n    model.eval();indices=ds.data[\'splits\'][split][rank::world];predictions=[]\n    for a in range(0,len(indices),cfg[\'eval_batch_docs\']):\n        batch=ds.batch(indices[a:a+cfg[\'eval_batch_docs\']],device)\n        with amp(device):\n            f=encode_all(model,batch,device,cfg[\'chunk_batch\']);z,_=model.classify(f,batch)\n        predictions.append(z.float().cpu().numpy())\n    z=np.concatenate(predictions) if predictions else np.zeros((0,cfg[\'label_count\']),np.float32)\n    local=Path(out)/f\'{split}_rank{rank}.npz\';np.savez(local,indices=indices,logits=z);barrier()\n    result=None\n    if rank==0:\n        byid={}\n        for r in range(world):\n            with np.load(Path(out)/f\'{split}_rank{r}.npz\') as saved:\n                byid.update(zip(saved[\'indices\'].tolist(),saved[\'logits\']))\n        expected=ds.data[\'splits\'][split]\n        if set(byid)!=set(expected):raise ValueError(\'Distributed inference missed/duplicated examples.\')\n        result=np.stack([byid[i] for i in expected]);np.save(Path(out)/f\'{split}_logits.npy\',result)\n    barrier();return result\n\ndef targets(ds,split):\n    values=[]\n    for i in ds.data[\'splits\'][split]:\n        d=ds.data[\'docs\'][i]\n        if d[\'gold\'] is None:raise ValueError(\'Cannot request test targets.\')\n        y=np.zeros(ds.data[\'label_count\'],np.uint8);y[d[\'gold\']]=1;values.append(y)\n    return np.stack(values)\n\ndef thresholds(y,p,support,shrink=25.):\n    """Exact per-label F1 cutoffs (tie-aware), shrunk in logit space toward one global cutoff."""\n    C=y.shape[1];active=np.asarray(support)>0\n    grid=np.r_[np.linspace(.01,.2,20),np.linspace(.225,.8,24)]\n    t=float(max(grid,key=lambda value:metric(y,p,value)[\'macro_f1\']))\n    result=np.full(C,t,dtype=np.float64)\n    logit=lambda x:np.log(np.clip(x,1e-5,1-1e-5)/np.clip(1-x,1e-5,1-1e-5))\n    for j in range(C):\n        positives=y[:,j].sum()\n        if not active[j]:result[j]=1.;continue\n        if positives<5:continue\n        order=np.argsort(-p[:,j],kind=\'stable\');sortedp=p[order,j]\n        true=np.cumsum(y[order,j]);k=np.arange(1,len(order)+1)\n        f=2*true/(k+positives);ends=np.r_[sortedp[:-1]!=sortedp[1:],True]\n        candidates=np.flatnonzero(ends);best=candidates[np.argmax(f[candidates])]\n        cut=float(np.clip(sortedp[best],1e-5,1-1e-5));strength=float(positives/(positives+shrink))\n        result[j]=float(probabilities((1-strength)*logit(t)+strength*logit(cut)))\n    return result\n\ndef monitor(y,p,support,seed):\n    order=np.random.default_rng(seed).permutation(len(y));middle=len(order)//2\n    calibration,selection=order[:middle],order[middle:]\n    if not len(calibration) or not len(selection):raise ValueError(\'Need at least two development cases.\')\n    t=thresholds(y[calibration],p[calibration],support)\n    return metric(y[selection],p[selection],t),t\n\ndef train(cfg,manifest,out,rank,world,device,smoke=False):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True);ds=Documents(manifest)\n    if ds.data[\'fingerprint\']!=cfg[\'data_fingerprint\']:raise ValueError(\'Data fingerprint changed.\')\n    torch.manual_seed(cfg[\'seed\']);model,provenance=build(cfg);model.to(device)\n    random.seed(cfg[\'seed\']+rank);np.random.seed(cfg[\'seed\']+rank);torch.manual_seed(cfg[\'seed\']+rank)\n    encoder=[];head=[];lexical=[]\n    for name,p in model.named_parameters():\n        (encoder if name.startswith(\'backbone.\') else lexical if name.startswith(\'lexical.\') else head).append(p)\n    optimizer=torch.optim.AdamW([{\'params\':encoder,\'lr\':cfg[\'encoder_lr\'],\'base_lr\':cfg[\'encoder_lr\'],\'weight_decay\':.01},\n        {\'params\':head,\'lr\':cfg[\'head_lr\'],\'base_lr\':cfg[\'head_lr\'],\'weight_decay\':.01},\n        {\'params\':lexical,\'lr\':cfg[\'lexical_lr\'],\'base_lr\':cfg[\'lexical_lr\'],\'weight_decay\':.05}],\n        fused=device.type==\'cuda\')\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=device.type==\'cuda\',init_scale=1024.)\n    support=np.asarray(ds.data[\'support\']);N=len(ds.data[\'splits\'][\'train\']);w=np.sqrt((N-support)/np.maximum(support,1)).clip(1,8)\n    normalizer=1+(support/N)*(w-1)\n    weights=torch.as_tensor(w,dtype=torch.float32,device=device);norm=torch.as_tensor(normalizer,dtype=torch.float32,device=device)\n    identity=digest({\'config\':cfg,\'world\':world,\'parent\':provenance[\'phase1_sha256\']})\n    state={\'identity\':identity,\'best_score\':-1.,\'best_epoch\':0,\'bad_epochs\':0,\'updates\':0,\'history\':[]}\n    epoch0=batch0=0;latest=out/\'latest.pt\'\n    if latest.exists() and not smoke:\n        old=torch.load(latest,map_location=\'cpu\',weights_only=False)\n        if old[\'state\'][\'identity\']!=identity:raise ValueError(\'Resume settings/world/parent changed. Use a new PROJECT.\')\n        model.load_state_dict(old[\'model\']);optimizer.load_state_dict(old[\'optimizer\']);scaler.load_state_dict(old[\'scaler\'])\n        epoch0,batch0=old[\'cursor\'];state=old[\'state\'];restore_rng(old[\'rng\'][rank],device);del old\n        if rank==0:print(\'Resuming at epoch, local batch:\',epoch0+1,batch0,flush=True)\n    if (out/\'complete.json\').exists() and not smoke:\n        if rank==0:print(\'Training already complete.\',flush=True)\n        return\n    batches_per_epoch=math.ceil(math.ceil(N/(world*cfg[\'batch_docs\']))/cfg[\'accumulation\'])\n    total_updates=cfg[\'epochs\']*batches_per_epoch;start=time.monotonic();optimizer.zero_grad(set_to_none=True)\n    for epoch in range(epoch0,cfg[\'epochs\']):\n        model.train();model.backbone.requires_grad_(epoch>=cfg[\'freeze_epochs\'] or smoke)\n        batches=epoch_batches(ds.data,cfg,epoch,rank,world)\n        stop=1 if smoke else len(batches)\n        for b in range(batch0,stop):\n            group_start=(b//cfg[\'accumulation\'])*cfg[\'accumulation\']\n            divisor=min(cfg[\'accumulation\'],stop-group_start)\n            batch=ds.batch(batches[b],device)\n            if smoke:\n                # Stress the configured encoder microbatch using long real TRAIN cases.\n                # Bounding this preflight only does not truncate actual training or inference.\n                chosen=sorted(ds.data[\'splits\'][\'train\'],key=lambda i:len(ds.data[\'docs\'][i][\'chunks\']),reverse=True)[:cfg[\'batch_docs\']]\n                batch=ds.batch(chosen,device);keep=[];counts=[];cursor=0\n                for n in batch[\'counts\']:\n                    take=min(n,cfg[\'chunk_batch\']);keep.extend(range(cursor,cursor+take));counts.append(take);cursor+=n\n                batch[\'chunks\']=[batch[\'chunks\'][i] for i in keep]\n                batch[\'position\']=batch[\'position\'][keep];batch[\'counts\']=counts\n            loss=cached_backward(model,batch,device,scaler,weights,norm,divisor)\n            if (b+1)%cfg[\'accumulation\']==0 or b+1==stop:\n                warm=max(1,int(.06*total_updates));u=state[\'updates\']\n                factor=min(1.,(u+1)/warm)*(.15+.85*.5*(1+math.cos(math.pi*min(1,u/max(1,total_updates)))))\n                for group in optimizer.param_groups:group[\'lr\']=group[\'base_lr\']*factor\n                sync_gradients(model,world);scaler.unscale_(optimizer)\n                g=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n                previous_scale=scaler.get_scale();scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True)\n                state[\'updates\']+=1\n                if rank==0 and (state[\'updates\']%cfg[\'log_every\']==0 or smoke):\n                    print(json.dumps({\'epoch\':epoch+1,\'batch\':b+1,\'updates\':state[\'updates\'],\'loss\':loss,\n                        \'gradient_norm\':float(g),\'amp_skipped\':scaler.get_scale()<previous_scale,\n                        \'elapsed_hours\':(time.monotonic()-start)/3600}),flush=True)\n                if smoke:\n                    probe=torch.stack([p.detach().float().sum() for p in model.parameters()])\n                    if world>1:\n                        lo=probe.clone();hi=probe.clone();dist.all_reduce(lo,op=dist.ReduceOp.MIN);dist.all_reduce(hi,op=dist.ReduceOp.MAX)\n                        if not torch.allclose(lo,hi,atol=1e-5,rtol=1e-5):raise ValueError(\'GPU ranks diverged.\')\n                    if not torch.isfinite(probe).all() or not math.isfinite(float(g)):raise ValueError(\'Nonfinite smoke update.\')\n                    if rank==0:atomic_json(out/\'smoke_pass.json\',{\'world_size\':world,\'device\':str(device),\'finite_update\':True,\n                        \'encoder_microbatch\':cfg[\'chunk_batch\'],\'preflight_chunks_per_rank\':len(batch[\'chunks\']),\n                        \'preflight_only_context_bound\':True,\n                        \'peak_reserved_gib\':torch.cuda.max_memory_reserved(device)/1024**3 if device.type==\'cuda\' else None})\n                    barrier();return\n                if state[\'updates\']%cfg[\'checkpoint_every\']==0:\n                    save(latest,model,cfg,provenance,optimizer,scaler,(epoch,b+1),state,rank,world,device)\n        batch0=0\n        logits=predict(model,ds,\'dev\',cfg,rank,world,device,out)\n        if rank==0:\n            y=targets(ds,\'dev\');p=probabilities(logits);p[:,support==0]=0\n            scores,_=monitor(y,p,support,cfg[\'seed\']);score=scores[\'macro_f1\']\n            # A warm-up-only checkpoint is not eligible as this is a full-fine-tuning candidate.\n            eligible=epoch>=cfg[\'freeze_epochs\']\n            improved=eligible and score>state[\'best_score\']+1e-6\n            state[\'history\'].append({\'epoch\':epoch+1,\'selection_half_metrics\':scores})\n            if improved:state.update(best_score=score,best_epoch=epoch+1,bad_epochs=0)\n            elif eligible:state[\'bad_epochs\']+=1\n            atomic_json(out/\'history.json\',state[\'history\'])\n            print(json.dumps({\'epoch\':epoch+1,\'held_out_dev_macro_f1\':score,\'best_epoch\':state[\'best_epoch\'],\n                \'elapsed_hours\':(time.monotonic()-start)/3600}),flush=True)\n        else:improved=None\n        if world>1:\n            objects=[state,improved];dist.broadcast_object_list(objects,src=0);state,improved=objects\n        if improved:save(out/\'best_model.pt\',model,cfg,provenance,rank=rank,world=world)\n        save(latest,model,cfg,provenance,optimizer,scaler,(epoch+1,0),state,rank,world,device)\n        if epoch+1>=cfg[\'min_epochs\'] and state[\'bad_epochs\']>=cfg[\'patience\']:break\n    if rank==0:atomic_json(out/\'complete.json\',{\'best_epoch\':state[\'best_epoch\'],\'selection_half_macro_f1\':state[\'best_score\'],\n        \'wall_hours_this_invocation\':(time.monotonic()-start)/3600,\'world_size\':world})\n    barrier()\n\ndef export(cfg,manifest,out,rank,world,device):\n    out=Path(out);ds=Documents(manifest);model,provenance=build(cfg,out/\'best_model.pt\');model.to(device)\n    z=predict(model,ds,\'dev\',cfg,rank,world,device,out)\n    if rank==0:\n        y=targets(ds,\'dev\');p=probabilities(z);support=np.asarray(ds.data[\'support\']);p[:,support==0]=0\n        threshold=thresholds(y,p,support)\n        atomic_json(out/\'calibration.json\',{\'threshold\':threshold.tolist(),\'dev_metrics_after_calibration\':metric(y,p,threshold),\n            \'note\':\'Final thresholds use all dev after checkpoint selection. Reported dev score is fitted, not a test estimate.\',\n            \'trainable_label_ids\':np.flatnonzero(support>0).tolist()})\n        rows=[{\'id\':ds.data[\'docs\'][i][\'id\'],\'gold\':ds.data[\'docs\'][i][\'gold\']} for i in ds.data[\'splits\'][\'dev\']]\n        atomic_json(out/\'dev_truth.json\',rows)\n        n=max(j for r in rows for j in r[\'gold\'])+1\n        labels=[[int(j) for j in np.flatnonzero(row>=threshold) if support[j]>0 and j<n] for row in p]\n        write_submission([r[\'id\'] for r in rows],labels,out/\'dev_submission.zip\')\n        with zipfile.ZipFile(out/\'dev_submission.zip\') as archive:(out/\'dev_prediction.json\').write_bytes(archive.read(\'prediction.json\'))\n    barrier()\n    # All fitting decisions and weights are now frozen; test has target=None.\n    z=predict(model,ds,\'test\',cfg,rank,world,device,out)\n    if rank==0:\n        cal=read_json(out/\'calibration.json\');p=probabilities(z);allowed=set(cal[\'trainable_label_ids\'])\n        labels=[[int(j) for j in np.flatnonzero(row>=np.asarray(cal[\'threshold\'])) if j in allowed] for row in p]\n        ids=[ds.data[\'docs\'][i][\'id\'] for i in ds.data[\'splits\'][\'test\']]\n        destination=Path(cfg[\'project\']).parent/\'phase5_one_model_submission.zip\'\n        rows,sha=write_submission(ids,labels,destination)\n        atomic_json(Path(cfg[\'project\'])/\'prediction.json\',rows)\n        atomic_json(out/\'submission_audit.json\',{\'test_count\':len(ids),\'prediction_sha256\':sha,\'submission\':str(destination),\n            \'one_model\':True,\'test_labels_used\':False,\'full_document_at_train_and_test\':True,\'provenance\':provenance})\n        print(\'FINAL SUBMISSION:\',destination,flush=True)\n    barrier()\n\ndef main():\n    parser=argparse.ArgumentParser();parser.add_argument(\'--config\',required=True);parser.add_argument(\'--action\',choices=[\'prepare\',\'smoke\',\'train\',\'export\'],required=True)\n    args=parser.parse_args();cfg=read_json(args.config);project=Path(cfg[\'project\'])\n    if args.action==\'prepare\':\n        path=prepare(cfg[\'data_root\'],cfg[\'tokenizer\'],project/\'cache\',cfg);data=read_json(path)\n        N=len(data[\'splits\'][\'train\']);support=np.asarray(data[\'support\']);w=np.sqrt((N-support)/np.maximum(support,1)).clip(1,8)\n        prevalence=(support+1)/(N+2);bias=np.log(prevalence*w/(1-prevalence))\n        cfg.update(data_fingerprint=data[\'fingerprint\'],label_count=data[\'label_count\'],sparse_dim=data[\'sparse_dim\'],\n            initial_bias=bias.tolist(),tokenizer_sha256=data[\'tokenizer_sha256\'])\n        atomic_json(args.config,cfg);return\n    rank,world,device=setup();out=project/\'run\';out.mkdir(parents=True,exist_ok=True)\n    manifest=project/\'cache/manifest.json\'\n    if args.action in (\'train\',\'smoke\'):train(cfg,manifest,out if args.action==\'train\' else project/\'preflight\',rank,world,device,args.action==\'smoke\')\n    else:export(cfg,manifest,out,rank,world,device)\n    if dist.is_initialized():dist.destroy_process_group()\nif __name__==\'__main__\':main()\n\n'
(CODE/'s5_run.py').write_text(SOURCE,encoding="utf-8")
compile(SOURCE,'s5_run.py',"exec")
print("Wrote",'s5_run.py')

In [ ]:
SOURCE = '"""Official Phase-5 input acquisition; no external pretrained weights."""\nimport os,shutil\nfrom pathlib import Path\nfrom lsi_data import read_json,atomic_json,file_hash,DATA_FILES\ndef obtain_data(project,local_root=None,revision=None):\n    destination=Path(project)/\'data\';destination.mkdir(parents=True,exist_ok=True);provenance=destination/\'source.json\'\n    if local_root:\n        source=Path(local_root)\n        if (source/\'Phase_5_LSI\').is_dir():source/=\'Phase_5_LSI\'\n        missing=[name for name in DATA_FILES if not (source/name).is_file()]\n        if missing:raise FileNotFoundError(\'Missing official Phase-5 files: \'+\', \'.join(missing))\n        origin={\'kind\':\'attached_official_phase5\',\'source\':str(source)}\n        for name in DATA_FILES:\n            target=destination/name;target.parent.mkdir(parents=True,exist_ok=True)\n            if target.resolve()!=(source/name).resolve():shutil.copy2(source/name,target)\n    else:\n        old=read_json(provenance) if provenance.exists() else None\n        if old and old.get(\'kind\')==\'official_huggingface\' and old.get(\'sha256\') and (revision is None or revision==old.get(\'revision\')):\n            if all((destination/name).is_file() for name in DATA_FILES):\n                hashes={name:file_hash(destination/name) for name in DATA_FILES}\n                if hashes!=old[\'sha256\']:raise ValueError(\'Cached official data changed; restore it or use a new PROJECT.\')\n                return str(destination)\n        from huggingface_hub import HfApi,hf_hub_download\n        token=os.environ.get(\'HF_TOKEN\')\n        if not token:\n            try:\n                from kaggle_secrets import UserSecretsClient\n                token=UserSecretsClient().get_secret(\'HF_TOKEN\')\n            except Exception:token=None\n        repo=\'Exploration-Lab/CS-781-Capstone\';commit=old[\'revision\'] if old and old.get(\'kind\')==\'official_huggingface\' else revision\n        try:\n            commit=HfApi(token=token).dataset_info(repo,revision=commit).sha\n            origin={\'kind\':\'official_huggingface\',\'repo\':repo,\'revision\':commit};atomic_json(provenance,origin)\n            for name in DATA_FILES:\n                target=destination/name;target.parent.mkdir(parents=True,exist_ok=True)\n                if not target.exists():\n                    source=hf_hub_download(repo,\'Phase_5_LSI/\'+name,repo_type=\'dataset\',revision=commit,token=token)\n                    shutil.copy2(source,target)\n        except Exception as error:raise RuntimeError(\'Attach Phase_5_LSI, or accept the official dataset terms and enable the HF_TOKEN Kaggle secret. Never paste API tokens into code.\') from error\n    origin[\'sha256\']={name:file_hash(destination/name) for name in DATA_FILES};atomic_json(provenance,origin)\n    return str(destination)\n'
(CODE/'s5_download.py').write_text(SOURCE,encoding="utf-8")
compile(SOURCE,'s5_download.py',"exec")
print("Wrote",'s5_download.py')

In [ ]:
SOURCE = '#!/usr/bin/env python3\n"""Local evaluator for Phase 5.\n\nTruth JSON: a list of objects, [{"id":"...", "gold":[0,2,5,...]}, ...]\nPrediction JSON: [{"id":"...", "pred":[0,2,5,...]}, ...]\n"""\nimport argparse,json,sys\nimport numpy as np\nfrom sklearn.metrics import accuracy_score, f1_score\n\ndef read_truth(path):\n    try:\n        x=json.load(open(path,encoding="utf-8"))\n    except json.JSONDecodeError as e:\n        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")\n    if not isinstance(x,list) or not x: raise ValueError(\'Truth must be a non-empty JSON list.\')\n    out={}; max_label=-1\n    for n,r in enumerate(x):\n        if not isinstance(r,dict) or \'id\' not in r or \'gold\' not in r: raise ValueError(f\'Truth item {n} must contain id and gold.\')\n        if not isinstance(r[\'id\'],str) or not r[\'id\'].strip() or not isinstance(r[\'gold\'],list): raise ValueError(f\'Invalid truth item {n}.\')\n        if r[\'id\'] in out: raise ValueError(f\'Duplicate truth ID: {r["id"]}\')\n        seen=set()\n        for label in r[\'gold\']:\n            if isinstance(label,bool) or not isinstance(label,int) or label<0 or label in seen: raise ValueError(f\'Invalid/duplicate label in truth item {n}.\')\n            seen.add(label); max_label=max(max_label,label)\n        out[r[\'id\']]=r[\'gold\']\n    return out,max_label+1\n\ndef read_pred(path,nlab):\n    x=json.load(open(path,encoding=\'utf-8\'))\n    if not isinstance(x,list) or not x: raise ValueError(\'Prediction must be a non-empty JSON list.\')\n    out={}\n    for n,r in enumerate(x):\n        if not isinstance(r,dict) or \'id\' not in r or \'pred\' not in r: raise ValueError(f\'Prediction item {n} must contain id and pred.\')\n        if not isinstance(r[\'id\'],str) or not isinstance(r[\'pred\'],list) or r[\'id\'] in out: raise ValueError(f\'Invalid or duplicate prediction item {n}.\')\n        seen=set()\n        for x in r[\'pred\']:\n            if isinstance(x,bool) or not isinstance(x,int) or x<0 or x>=nlab or x in seen: raise ValueError(f\'Invalid/duplicate label in prediction item {n}.\')\n            seen.add(x)\n        out[r[\'id\']]=r[\'pred\']\n    return out\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 5 multilabel predictions on your own held-out split.\')\n    ap.add_argument(\'--truth\',required=True); ap.add_argument(\'--prediction\',required=True); a=ap.parse_args()\n    t,n=read_truth(a.truth); p=read_pred(a.prediction,n)\n    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")\n    yt=np.zeros((len(t),n),dtype=np.uint8); yp=np.zeros((len(t),n),dtype=np.uint8)\n    for i,k in enumerate(t):\n        for x in t[k]: yt[i,x]=1\n        for x in p[k]: yp[i,x]=1\n    print(f\'Examples: {len(t)}\'); print(f\'Labels: {n}\')\n    print(f\'Macro-F1: {f1_score(yt,yp,average="macro",zero_division=0):.6f}\')\n    micro=f1_score(yt,yp,average=\'micro\',zero_division=0)\n    subset=accuracy_score(yt,yp)\n    print(f\'Micro-F1: {micro:.6f}\')\n    print(f\'Subset Accuracy: {subset:.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\',file=sys.stderr); sys.exit(1)\n'
(CODE/'evaluate_phase5_official.py').write_text(SOURCE,encoding="utf-8")
compile(SOURCE,'evaluate_phase5_official.py',"exec")
print("Wrote",'evaluate_phase5_official.py')

In [ ]:
sys.path.insert(0,str(CODE))
from s5_download import obtain_data
from lsi_data import read_json,atomic_json,file_hash
data_root=obtain_data(PROJECT,DATA_ROOT,DATA_REVISION)
checkpoint=PHASE1_ROOT/'runs/main_decoder/best_model.pt'
tokenizer=PHASE1_ROOT/'tokenizer.json'
for path in (checkpoint,tokenizer):
    if not path.is_file():raise FileNotFoundError(f'Attach phase-1-outputs: missing {path}')
CONFIG=PROJECT/'config.json'
requested={**SETTINGS,'project':str(PROJECT),'phase1_checkpoint':str(checkpoint),'tokenizer':str(tokenizer),'data_root':str(data_root),
           'code_sha256':{p.name:file_hash(p) for p in sorted(CODE.glob('*.py'))}}
if CONFIG.exists():
    saved=read_json(CONFIG)
    old_code=saved.get('code_sha256',{})
    expected_old={**requested['code_sha256'],'s5_model.py':'219ecd82f988788dcf1e29e2cf9d729fecbd5c0b579d0b8634090f510e0df211'}
    same_settings=all(saved.get(k)==v for k,v in requested.items() if k!='code_sha256')
    if same_settings and old_code==expected_old and old_code!=requested['code_sha256']:
        if any((PROJECT/'run'/name).exists() for name in ('latest.pt','best_model.pt')):
            raise ValueError('A trained checkpoint exists; use a fresh PROJECT for the precision repair.')
        saved['code_sha256']=requested['code_sha256'];atomic_json(CONFIG,saved)
        print('Applied FP16 attention repair; reusing the prepared token and sparse caches.')
    if any(saved.get(k)!=v for k,v in requested.items()):
        raise ValueError('Configuration changed. Restore the old settings or choose a new PROJECT.')
else:atomic_json(CONFIG,requested)

def launch(action,gpus=1):
    script=CODE/'s5_run.py'
    args=[str(script),'--action',action,'--config',str(CONFIG)]
    if gpus==2:
        cmd=[sys.executable,'-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2',*args]
    else:cmd=[sys.executable,'-u',*args]
    env=dict(os.environ);env['PYTHONPATH']=str(CODE)+os.pathsep+env.get('PYTHONPATH','')
    log=PROJECT/f'{action}.log'
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
        for line in process.stdout:print(line,end='',flush=True);handle.write(line);handle.flush()
        if process.wait()!=0:raise RuntimeError(f'{action} failed. Read {log}; completed checkpoints are retained.')
launch('prepare')
data=read_json(PROJECT/'cache/manifest.json')
print('Cases:',{s:len(v) for s,v in data['splits'].items()})
print('All-token count:',data['token_count'],'all chunks:',data['chunk_count'])
print('Label dimension:',data['label_count'],'sparse feature dimension:',data['sparse_dim'])
print('Labels with no train positives:',[i for i,n in enumerate(data['support']) if n==0])
atomic_json(PROJECT/'data_audit.json',data['audit'])
del data

## Actual two-GPU preflight

This performs one real finite optimizer update on both GPUs and checks that the model copies remain synchronized. It uses long actual training cases to exercise the configured encoder microbatch, bounding each preflight case to `chunk_batch` windows. This bound applies only to the preflight: actual training and inference retain every window. The cell writes only preflight artifacts; real training starts again from the original Phase-1 weights.

Only CPU verification was possible while building this notebook. **The code has not been run on T4 hardware here.** This cell must pass in your Kaggle session before training.

In [ ]:
launch('smoke',gpus=2)
smoke=read_json(PROJECT/'preflight/smoke_pass.json')
assert smoke['world_size']==2 and smoke['finite_update']
print('PASS actual two-GPU update:',smoke)

## Train the single candidate

The first epoch warms up the new head and lexical residual with the foundation frozen. All foundation parameters are trainable from epoch 2. A warm-up-only checkpoint cannot be selected as the final model. The loss is a capped positive-weighted binary cross-entropy plus a 0.2-weight neural-only auxiliary loss, keeping the neural path useful while the sparse residual learns.

The dev split is deterministically halved: one half fits epoch-monitoring thresholds; the other half selects the checkpoint. Early stopping starts after the configured minimum epochs. No test labels enter either half. After selecting the checkpoint, final thresholds are fitted on the complete dev split with shrinkage for rare labels. The resulting full-dev metric is a calibrated training diagnostic, not an unbiased test estimate.

Training checkpoints include optimizer, scaler, accumulated-update cursor and per-rank random state. Resume saves occur only after complete updates. Re-run this cell with the same inputs/settings to resume; up to 199 updates since the last save can be repeated. Do not modify the modules or GPU count midway through a run.

**Runtime:** full-document adaptation is more work per epoch than the prior three-window model. Running one candidate avoids repeating seven encoders, feature banks and heads. The actual epoch logs are the reliable ETA source; no fixed total time is asserted.

In [ ]:
launch('train',gpus=2)
summary=read_json(PROJECT/'run/complete.json')
print(json.dumps(summary,indent=2))

## Freeze decisions, predict test, validate and create one submission

The selected model generates dev probabilities for final threshold calibration, then test predictions. Every test document is encoded in full. There is no test scoring, leaderboard-driven search, output from previous candidates or forced label-count heuristic.

The official evaluator is run on dev. Its Macro-F1 dimension includes integer ID gaps through the largest reference label. If a mapped label has no training positives, the model cannot learn a supervised predictor for it and suppresses it; your supplied executed run reported no such missing labels.

In [ ]:
launch('export',gpus=2)
result=subprocess.run([sys.executable,str(CODE/'evaluate_phase5_official.py'),
    '--truth',str(PROJECT/'run/dev_truth.json'),'--prediction',str(PROJECT/'run/dev_prediction.json')],
    text=True,capture_output=True,check=True)
print(result.stdout)
cal=read_json(PROJECT/'run/calibration.json')
for label,key in [('Macro-F1','macro_f1'),('Micro-F1','micro_f1'),('Subset Accuracy','subset_accuracy')]:
    actual=float(next(line.split(':')[1] for line in result.stdout.splitlines() if line.startswith(label+':')))
    assert abs(actual-cal['dev_metrics_after_calibration'][key])<1e-6,(label,actual)
SUBMISSION=WORKING/'phase5_one_model_submission.zip'
with zipfile.ZipFile(SUBMISSION) as archive:
    assert archive.namelist()==['prediction.json']
    prediction=json.loads(archive.read('prediction.json'))
manifest=read_json(PROJECT/'cache/manifest.json')
expected=[manifest['docs'][i]['id'] for i in manifest['splits']['test']]
allowed=set(cal['trainable_label_ids'])
assert len(expected)==len(set(expected))
assert [r['id'] for r in prediction]==expected
assert all(set(r)=={'id','pred'} and isinstance(r['pred'],list)
           and len(r['pred'])==len(set(r['pred']))
           and all(type(j) is int and j in allowed for j in r['pred']) for r in prediction)
print('Verified test cases:',len(prediction),'mean predicted label count:',sum(len(r['pred']) for r in prediction)/len(prediction))
print('Submit THIS file:',SUBMISSION)
display(FileLink(str(SUBMISSION)))

## Save the one model for later inference

This archive contains the single best full model, thresholds, sparse vectorizers, tokenizer, configuration and runtime code. The full model checkpoint contains the foundation weights too, so restoring it does not require access to the original Phase-1 checkpoint. It omits raw data, token/sparse caches and optimizer recovery state from the download; these remain in the Kaggle output.

To reconstruct the model after extracting, add the archive's `code/` directory to `sys.path`, load `config.json`, and call `s5_run.build(config, "run/best_model.pt")`. Checkpoint-based restoration ignores the old Phase-1 path. Processing new documents must still use the saved tokenizer/vectorizers and the identical chunking/pooling rules. No second model is trained by this cell.

In [ ]:
MODEL_ARCHIVE=WORKING/'phase5_one_model_weights.zip'
files=[PROJECT/'run/best_model.pt',PROJECT/'run/calibration.json',CONFIG,
       PROJECT/'cache/vectorizers.joblib',PROJECT/'run/submission_audit.json',PROJECT/'run/complete.json',
       PROJECT/'run/history.json',PROJECT/'data_audit.json']
files+=sorted(CODE.glob('*.py'))
need=sum(p.stat().st_size for p in files)+tokenizer.stat().st_size
if shutil.disk_usage(WORKING).free<need*1.1:
    print('Submission is ready. Skipping the optional model archive because free disk is low.')
else:
    with zipfile.ZipFile(MODEL_ARCHIVE,'w',zipfile.ZIP_STORED) as archive:
        for p in files:archive.write(p,str(p.relative_to(PROJECT)))
        archive.write(tokenizer,'tokenizer.json')
    display(FileLink(str(MODEL_ARCHIVE)))
display(FileLink(str(SUBMISSION)))

## Verification completed before delivery

Synthetic CPU tests checked: every token exactly once; train-only sparse vocabulary; test-label changes cannot alter parsed model data; every parameter's cached gradient agrees with ordinary full-document backpropagation, including loss scaling; accumulated shard gradients match the full document batch; token attention across chunks equals direct attention over the concatenated token states; padding invariance; the model can fit a small fixture; full training/export; checkpoint restoration without the original foundation path; interrupted/resumed weights and optimizer match an uninterrupted run bit for bit, including dropout; one ZIP with correct original IDs; and official evaluator parity.

**FP16 repair:** token and document attention scoring, masking and pooling now run inside explicit FP32 subregions. The encoder retains AMP. This prevents the `c10::Half` padding-mask overflow observed in the uploaded T4 preflight. Existing prepared caches are compatible; the setup cell recognizes the exact old module fingerprint and upgrades it before training. The repair was checked with CPU FP16 autocast, including forward and backward; T4 verification still runs in the preflight cell above. See [PyTorch AMP subregion guidance](https://docs.pytorch.org/docs/2.11/amp.html).

No competition score, actual T4 throughput or successful two-GPU run is claimed from those tests. The preflight above verifies GPU execution in your environment. Your existing leaderboard score is the baseline; the new submission must be evaluated to establish whether this candidate improves it.